# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through the process of loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata
metadata = dataset.metadata
print(f"Dataset: {getattr(metadata, 'name', None)}\n\nDescription: {getattr(metadata, 'description', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id

record_set_objs = list(dataset.record_sets())
print(f"Number of record sets: {len(record_set_objs)}\n")

for rset in record_set_objs:
    print(f"Record Set: {rset.id}")
    print(f"  Name: {getattr(rset, 'name', '')}")
    print(f"  Description: {getattr(rset, 'description', '')}")
    print("  Fields:")
    for field in rset.fields:
        print(f"    {field.id}: {field.name}")
    print('---')


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Record set and field `@id`s are shown above for reference.

In [ ]:
# Select record sets to extract (identified by their @id)
record_set_ids = [rset.id for rset in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded {len(df)} records for record set {record_set_id}")
        print(df.head())
    else:
        print(f"\nNo records found for record set {record_set_id}")

# For demonstration, pick the first non-empty record set
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break
if main_record_set_id:
    print(f"\nColumns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular record sets found.")


## 4. Exploratory Data Analysis (EDA)
Let's apply some basic EDA: filter, normalize, and group by fields.

Choose a numeric field and a potential grouping field from the above record set.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Try to automatically select a numeric field (float/integers)
    # If not available, update the field manually after inspecting columns
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].dtype in ['float64', 'int64'] else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to select a categorical/group field
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break

        if group_field:
            # Group by and compute mean normalized value
            grouped_df = filtered_df.groupby(group_field)[f"{numeric_field}_normalized"].mean().reset_index()
            print(f"\nGrouped mean of normalized {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No string/categorical column found to group by.")
    else:
        print("No numeric field found to use for filtering and normalization.")
else:
    print("No data available for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field:
    # Plot distribution of the numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df is available and group_field found, plot it too
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(12, 5))
        sns.barplot(data=grouped_df, x=group_field, y=f"{numeric_field}_normalized")
        plt.title(f"Mean normalized {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean normalized {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough numeric data found for plotting.")

## 6. Conclusion
In this notebook, we have demonstrated how to load, inspect, and process a complex dataset defined by a Croissant schema using the `mlcroissant` library. We:
- Loaded the dataset and inspected its structure.
- Explored available record sets and their fields by their `@id` identifiers.
- Loaded data into pandas DataFrames and performed basic filtering, normalization, and grouping.
- Visualized numeric data distributions and group differences.

For deeper analysis, explore each record set and field as necessary by referencing their `@id` as shown above. This workflow is generalizable to any FAIR-compliant Croissant dataset using `mlcroissant`.